# Axolotl GRPO: Tool-Use RL Training
**Finetuning Agentic Workflows with RL — Colab Pro+**

Uses Axolotl framework for GRPO training on tool-call generation.
Same recipe as MATH experiments but applied to agentic tool use.

In [ ]:
# Install Axolotl
!pip install -q axolotl[flash-attn] 2>/dev/null || !pip install -q axolotl
!pip install -q wandb

In [ ]:
import torch
print(f"GPU: {torch.cuda.get_device_name(0)}")
print(f"Memory: {torch.cuda.get_device_properties(0).total_mem / 1e9:.1f} GB")
!nvidia-smi

In [ ]:
# Clone the experiment repo for configs
!git clone https://github.com/arvindcr4/tinker-atropos.git /content/tinker-atropos 2>/dev/null || echo "Already cloned"
!ls /content/tinker-atropos/configs/

In [ ]:
%%writefile /content/tool_use_dataset.jsonl
{"instruction": "You are a helpful assistant that uses tools. Respond with a JSON tool call.", "input": "What's the weather like in London?", "output": "{\"tool\": \"get_weather\", \"arguments\": {\"location\": \"London\", \"unit\": \"celsius\"}}"}
{"instruction": "You are a helpful assistant that uses tools. Respond with a JSON tool call.", "input": "Calculate 15 * 23 + 7", "output": "{\"tool\": \"calculator\", \"arguments\": {\"expression\": \"15 * 23 + 7\"}}"}
{"instruction": "You are a helpful assistant that uses tools. Respond with a JSON tool call.", "input": "Set a timer for 25 minutes for my workout", "output": "{\"tool\": \"set_timer\", \"arguments\": {\"duration_minutes\": 25, \"label\": \"workout\"}}"}
{"instruction": "You are a helpful assistant that uses tools. Respond with a JSON tool call.", "input": "Convert 100 kilometers to miles", "output": "{\"tool\": \"unit_converter\", \"arguments\": {\"value\": 100, \"from_unit\": \"kilometers\", \"to_unit\": \"miles\"}}"}
{"instruction": "You are a helpful assistant that uses tools. Respond with a JSON tool call.", "input": "Translate 'Hello, how are you?' to Spanish", "output": "{\"tool\": \"translator\", \"arguments\": {\"text\": \"Hello, how are you?\", \"source_lang\": \"English\", \"target_lang\": \"Spanish\"}}"}
{"instruction": "You are a helpful assistant that uses tools. Respond with a JSON tool call.", "input": "Search for Python tutorials", "output": "{\"tool\": \"web_search\", \"arguments\": {\"query\": \"Python tutorials\", \"num_results\": 5}}"}
{"instruction": "You are a helpful assistant that uses tools. Respond with a JSON tool call.", "input": "Send an email to team@company.com about the meeting", "output": "{\"tool\": \"send_email\", \"arguments\": {\"to\": \"team@company.com\", \"subject\": \"Meeting Update\", \"body\": \"The meeting has been rescheduled.\"}}"}
{"instruction": "You are a helpful assistant that uses tools. Respond with a JSON tool call.", "input": "List files in /home/user/projects", "output": "{\"tool\": \"file_manager\", \"arguments\": {\"action\": \"list\", \"path\": \"/home/user/projects\"}}"}
{"instruction": "You are a helpful assistant that uses tools. Respond with a JSON tool call.", "input": "Remind me to call the dentist at 2pm", "output": "{\"tool\": \"reminder\", \"arguments\": {\"message\": \"Call the dentist\", \"time\": \"14:00\"}}"}
{"instruction": "You are a helpful assistant that uses tools. Respond with a JSON tool call.", "input": "Create a note titled 'Ideas'", "output": "{\"tool\": \"note_taker\", \"arguments\": {\"action\": \"create\", \"title\": \"Ideas\", \"content\": \"New project brainstorm\"}}"}
{"instruction": "You are a helpful assistant that uses tools. Respond with a JSON tool call.", "input": "Convert 500 euros to US dollars", "output": "{\"tool\": \"currency_converter\", \"arguments\": {\"amount\": 500, \"from_currency\": \"EUR\", \"to_currency\": \"USD\"}}"}
{"instruction": "You are a helpful assistant that uses tools. Respond with a JSON tool call.", "input": "Find a chicken stir fry recipe", "output": "{\"tool\": \"recipe_finder\", \"arguments\": {\"query\": \"chicken stir fry\", \"cuisine\": \"Asian\"}}"}
{"instruction": "You are a helpful assistant that uses tools. Respond with a JSON tool call.", "input": "Generate a 20-character password with symbols", "output": "{\"tool\": \"password_generator\", \"arguments\": {\"length\": 20, \"include_symbols\": true}}"}
{"instruction": "You are a helpful assistant that uses tools. Respond with a JSON tool call.", "input": "What's the stock price of AAPL?", "output": "{\"tool\": \"stock_price\", \"arguments\": {\"ticker\": \"AAPL\"}}"}
{"instruction": "You are a helpful assistant that uses tools. Respond with a JSON tool call.", "input": "Define the word 'ubiquitous'", "output": "{\"tool\": \"dictionary_lookup\", \"arguments\": {\"word\": \"ubiquitous\"}}"}
{"instruction": "You are a helpful assistant that uses tools. Respond with a JSON tool call.", "input": "Weather in Tokyo in fahrenheit?", "output": "{\"tool\": \"get_weather\", \"arguments\": {\"location\": \"Tokyo\", \"unit\": \"fahrenheit\"}}"}
{"instruction": "You are a helpful assistant that uses tools. Respond with a JSON tool call.", "input": "Calculate sqrt(144)", "output": "{\"tool\": \"calculator\", \"arguments\": {\"expression\": \"sqrt(144)\"}}"}
{"instruction": "You are a helpful assistant that uses tools. Respond with a JSON tool call.", "input": "Set a 5 minute timer for tea", "output": "{\"tool\": \"set_timer\", \"arguments\": {\"duration_minutes\": 5, \"label\": \"tea\"}}"}
{"instruction": "You are a helpful assistant that uses tools. Respond with a JSON tool call.", "input": "Convert 72 fahrenheit to celsius", "output": "{\"tool\": \"unit_converter\", \"arguments\": {\"value\": 72, \"from_unit\": \"fahrenheit\", \"to_unit\": \"celsius\"}}"}
{"instruction": "You are a helpful assistant that uses tools. Respond with a JSON tool call.", "input": "Translate 'Thank you' to Japanese", "output": "{\"tool\": \"translator\", \"arguments\": {\"text\": \"Thank you\", \"source_lang\": \"English\", \"target_lang\": \"Japanese\"}}"}
{"instruction": "You are a helpful assistant that uses tools. Respond with a JSON tool call.", "input": "Search for AI news 2026", "output": "{\"tool\": \"web_search\", \"arguments\": {\"query\": \"AI news 2026\", \"num_results\": 10}}"}
{"instruction": "You are a helpful assistant that uses tools. Respond with a JSON tool call.", "input": "Email john@example.com about the deadline", "output": "{\"tool\": \"send_email\", \"arguments\": {\"to\": \"john@example.com\", \"subject\": \"Project Deadline\", \"body\": \"Reminder about the upcoming deadline.\"}}"}
{"instruction": "You are a helpful assistant that uses tools. Respond with a JSON tool call.", "input": "Read /home/user/notes.txt", "output": "{\"tool\": \"file_manager\", \"arguments\": {\"action\": \"read\", \"path\": \"/home/user/notes.txt\"}}"}
{"instruction": "You are a helpful assistant that uses tools. Respond with a JSON tool call.", "input": "Remind me to submit the report by 5pm", "output": "{\"tool\": \"reminder\", \"arguments\": {\"message\": \"Submit the report\", \"time\": \"17:00\"}}"}
{"instruction": "You are a helpful assistant that uses tools. Respond with a JSON tool call.", "input": "How much is 1000 GBP in INR?", "output": "{\"tool\": \"currency_converter\", \"arguments\": {\"amount\": 1000, \"from_currency\": \"GBP\", \"to_currency\": \"INR\"}}"}
{"instruction": "You are a helpful assistant that uses tools. Respond with a JSON tool call.", "input": "Find an Italian pasta recipe", "output": "{\"tool\": \"recipe_finder\", \"arguments\": {\"query\": \"pasta\", \"cuisine\": \"Italian\"}}"}
{"instruction": "You are a helpful assistant that uses tools. Respond with a JSON tool call.", "input": "Generate a 12 character password without symbols", "output": "{\"tool\": \"password_generator\", \"arguments\": {\"length\": 12, \"include_symbols\": false}}"}
{"instruction": "You are a helpful assistant that uses tools. Respond with a JSON tool call.", "input": "TSLA stock price?", "output": "{\"tool\": \"stock_price\", \"arguments\": {\"ticker\": \"TSLA\"}}"}
{"instruction": "You are a helpful assistant that uses tools. Respond with a JSON tool call.", "input": "What does 'ephemeral' mean?", "output": "{\"tool\": \"dictionary_lookup\", \"arguments\": {\"word\": \"ephemeral\"}}"}
{"instruction": "You are a helpful assistant that uses tools. Respond with a JSON tool call.", "input": "Weather in New York City?", "output": "{\"tool\": \"get_weather\", \"arguments\": {\"location\": \"New York City\", \"unit\": \"fahrenheit\"}}"}
{"instruction": "You are a helpful assistant that uses tools. Respond with a JSON tool call.", "input": "What is 2^10?", "output": "{\"tool\": \"calculator\", \"arguments\": {\"expression\": \"2^10\"}}"}

In [ ]:
%%writefile /content/axolotl_tool_use_grpo.yaml
# Axolotl GRPO config for tool-use training
# Same recipe as MATH experiments

base_model: Qwen/Qwen2-0.5B-Instruct
model_type: AutoModelForCausalLM
tokenizer_type: AutoTokenizer
trust_remote_code: true

# RL method
rl: grpo

# Dataset
datasets:
  - path: /content/tool_use_dataset.jsonl
    type: alpaca

# LoRA config — matches TINKER experiments
adapter: lora
lora_r: 32
lora_alpha: 64
lora_dropout: 0.05
lora_target_modules:
  - q_proj
  - k_proj
  - v_proj
  - o_proj
  - gate_proj
  - up_proj
  - down_proj

# Training hyperparameters — match TINKER recipe
sequence_len: 512
sample_packing: false
pad_to_sequence_len: true

num_epochs: 5
micro_batch_size: 4
gradient_accumulation_steps: 8
learning_rate: 0.00004
lr_scheduler: cosine
warmup_steps: 10
weight_decay: 0.01
optimizer: adamw_torch

# GRPO specific
grpo_num_generations: 16
grpo_max_completion_length: 256
grpo_temperature: 0.7

# Precision
bf16: true
tf32: true

# Output
output_dir: /content/tool_use_grpo_output
logging_steps: 1
save_steps: 10
eval_steps: 10
save_total_limit: 3

# Tracking
wandb_project: tinker-rl-agentic
wandb_run_id: tool-use-grpo-colab

# Hub
hub_model_id: arvindcr4/tool-call-grpo-qwen0.5b
push_dataset_to_hub: false

In [ ]:
# Login to HuggingFace and Wandb
from huggingface_hub import login
login()

import wandb
wandb.login()

In [ ]:
# Run Axolotl training
!accelerate launch -m axolotl.cli.train /content/axolotl_tool_use_grpo.yaml

In [ ]:
# If GRPO is not supported in your axolotl version, fall back to SFT first then GRPO via TRL
# Alternative: run with TRL directly
!accelerate launch -m axolotl.cli.train /content/axolotl_tool_use_grpo.yaml 2>&1 || \
  echo "\n\nIf GRPO failed, run the TRL notebook (colab_tool_use_grpo.ipynb) instead"

In [ ]:
# Push to hub
!python -m axolotl.cli.merge_lora /content/axolotl_tool_use_grpo.yaml --lora_model_dir /content/tool_use_grpo_output
!huggingface-cli upload arvindcr4/tool-call-grpo-qwen0.5b /content/tool_use_grpo_output/merged

## Evaluation
Compare base model vs GRPO-trained model on held-out tool-use prompts

In [ ]:
import json
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer
from peft import PeftModel

TOOL_NAMES = {"get_weather", "calculator", "set_timer", "unit_converter", "translator",
              "web_search", "send_email", "file_manager", "reminder", "note_taker",
              "currency_converter", "recipe_finder", "password_generator", "stock_price", "dictionary_lookup"}

SYSTEM = """You are a helpful assistant that uses tools. Respond with ONLY a JSON tool call: {"tool": "<name>", "arguments": {...}}"""

EVAL_PROMPTS = [
    ("Convert 5 miles to km", "unit_converter"),
    ("What does 'ephemeral' mean?", "dictionary_lookup"),
    ("Tesla stock price?", "stock_price"),
    ("Translate 'Good morning' to French", "translator"),
    ("Remind me at 5pm about groceries", "reminder"),
    ("Timer for 10 min meditation", "set_timer"),
    ("Note: Meeting agenda items", "note_taker"),
    ("Search web for PyTorch tutorials", "web_search"),
    ("Email alice@co.com project update", "send_email"),
    ("List files in ~/Documents", "file_manager"),
    ("Convert 200 EUR to JPY", "currency_converter"),
    ("Find Thai curry recipe", "recipe_finder"),
    ("Generate 16 char password", "password_generator"),
    ("Weather in Paris?", "get_weather"),
    ("Calculate 2^16", "calculator"),
]

def eval_model(model, tokenizer):
    correct = 0
    for prompt, expected in EVAL_PROMPTS:
        msgs = [{"role": "system", "content": SYSTEM}, {"role": "user", "content": prompt}]
        text = tokenizer.apply_chat_template(msgs, tokenize=False, add_generation_prompt=True)
        inputs = tokenizer(text, return_tensors="pt").to(model.device)
        with torch.no_grad():
            out = model.generate(**inputs, max_new_tokens=256, do_sample=False,
                                pad_token_id=tokenizer.pad_token_id)
        resp = tokenizer.decode(out[0][inputs["input_ids"].shape[1]:], skip_special_tokens=True).strip()
        try:
            p = json.loads(resp)
            ok = p.get("tool") == expected
        except:
            ok = False
        status = "PASS" if ok else "FAIL"
        correct += int(ok)
        print(f"  {status} {prompt[:35]:35s} expected={expected:20s} got={resp[:60]}")
    return correct / len(EVAL_PROMPTS)

# Base model
print("=" * 60)
print("BASE MODEL")
print("=" * 60)
base = AutoModelForCausalLM.from_pretrained("Qwen/Qwen2-0.5B-Instruct", torch_dtype=torch.bfloat16, device_map="auto", trust_remote_code=True)
tok = AutoTokenizer.from_pretrained("Qwen/Qwen2-0.5B-Instruct", trust_remote_code=True)
if tok.pad_token is None: tok.pad_token = tok.eos_token
base_acc = eval_model(base, tok)
del base; torch.cuda.empty_cache()

# GRPO model
print("\n" + "=" * 60)
print("GRPO MODEL")
print("=" * 60)
grpo = AutoModelForCausalLM.from_pretrained("Qwen/Qwen2-0.5B-Instruct", torch_dtype=torch.bfloat16, device_map="auto", trust_remote_code=True)
grpo = PeftModel.from_pretrained(grpo, "/content/tool_use_grpo_output")
grpo_acc = eval_model(grpo, tok)

print(f"\n{'='*60}")
print(f"Base: {base_acc:.1%} | GRPO: {grpo_acc:.1%} | Delta: {grpo_acc-base_acc:+.1%}")
print(f"{'='*60}")